In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0


create tensor

In [2]:
x = torch.tensor([1, 2, 3, 4])

print(x)
print(x.shape)
print(x.dtype)

tensor([1, 2, 3, 4])
torch.Size([4])
torch.int64


2d

In [3]:
x = torch.tensor([
    [1, 2],
    [3, 4],
    [5, 6]
])

print(x)
print(x.shape)

tensor([[1, 2],
        [3, 4],
        [5, 6]])
torch.Size([3, 2])


model will learn 
If x1 + x2 > 0 → Class 1
If x1 + x2 <= 0 → Class 0

In [4]:
torch.manual_seed(42)

X = torch.randn(1000, 2)

y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([1000, 2])
y shape: torch.Size([1000, 1])


In [5]:
train_size = int(0.8 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: torch.Size([800, 2])
Testing: torch.Size([200, 2])


Train and Split

In [6]:
train_size = int(0.8 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: torch.Size([800, 2])
Testing: torch.Size([200, 2])


Data Loader


In [8]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [11]:
X_batch, y_batch = next(iter(train_loader))

print("X batch:", X_batch.shape)
print("y batch:", y_batch.shape)

X batch: torch.Size([32, 2])
y batch: torch.Size([32, 1])


Build the neural network model 

In [12]:
class BinaryClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(2, 8)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)

        return x

/create the model

In [13]:
model = BinaryClassifier()

print(model)

BinaryClassifier(
  (layer1): Linear(in_features=2, out_features=8, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=8, out_features=1, bias=True)
)


Forward pass

In [14]:
output = model(X_batch)

print(output.shape)

torch.Size([32, 1])


losss function


In [15]:
loss_fn = nn.BCEWithLogitsLoss()

In [16]:
loss_fn = nn.BCEWithLogitsLoss()

output = model(X_batch)

loss = loss_fn(output, y_batch)

print("Loss:", loss.item())

Loss: 0.6684539318084717


Auto grad

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

y.backward()

print("Gradient:", x.grad)

Gradient: tensor(4.)


In [18]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

y.backward()

print("Gradient:", x.grad)

Gradient: tensor(4.)


Backpropagation

In [19]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

In [21]:
optimizer.zero_grad()

In [22]:
output = model(X_batch)

In [24]:
loss = loss_fn(output, y_batch)

In [26]:
loss.backward()

In [27]:
optimizer.step()

Training

In [28]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # 1. Clear old gradients
        optimizer.zero_grad()

        # 2. Forward pass
        output = model(X_batch)

        # 3. Calculate loss
        loss = loss_fn(output, y_batch)

        # 4. Backpropagation
        loss.backward()

        # 5. Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {average_loss:.4f}"
    )

Epoch 1/20, Loss: 0.6002
Epoch 2/20, Loss: 0.4834
Epoch 3/20, Loss: 0.3743
Epoch 4/20, Loss: 0.2819
Epoch 5/20, Loss: 0.2161
Epoch 6/20, Loss: 0.1744
Epoch 7/20, Loss: 0.1478
Epoch 8/20, Loss: 0.1298
Epoch 9/20, Loss: 0.1170
Epoch 10/20, Loss: 0.1071
Epoch 11/20, Loss: 0.0994
Epoch 12/20, Loss: 0.0933
Epoch 13/20, Loss: 0.0879
Epoch 14/20, Loss: 0.0839
Epoch 15/20, Loss: 0.0802
Epoch 16/20, Loss: 0.0769
Epoch 17/20, Loss: 0.0739
Epoch 18/20, Loss: 0.0713
Epoch 19/20, Loss: 0.0689
Epoch 20/20, Loss: 0.0669


In [29]:
model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        output = model(X_batch)

        predictions = (torch.sigmoid(output) >= 0.5).float()

In [30]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        output = model(X_batch)

        probabilities = torch.sigmoid(output)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == y_batch).sum().item()

        total += y_batch.size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 98.50%


New prediction

In [31]:
new_data = torch.tensor([
    [2.0, 3.0],
    [-2.0, -3.0],
    [4.0, -1.0],
    [-4.0, 1.0]
])

In [32]:
model.eval()

with torch.no_grad():

    output = model(new_data)

    probabilities = torch.sigmoid(output)

    predictions = (probabilities >= 0.5).float()

print("Probabilities:")
print(probabilities)

print("Predictions:")
print(predictions)

Probabilities:
tensor([[1.0000e+00],
        [3.9631e-13],
        [9.9995e-01],
        [2.8926e-08]])
Predictions:
tensor([[1.],
        [0.],
        [1.],
        [0.]])
